# INFERENCE COST

E-MoE replaces the MLP of every DiT block by $K=8$ experts, so it has $\approx 3.8\times$ more parameters than MDLM, but at inference every
token is processed by exactly one expert. Here you can find an ablation on the compute per denoising step and for the whole generation and see that this is the same for all models: MDLM, MDLM-MoE, E-MoE.

In [1]:
import os, sys
import pandas as pd
import torch
import omegaconf, hydra
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from torch.utils.flop_counter import FlopCounterMode

REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, REPO)
for name, fn in [('cwd', os.getcwd), ('device_count', torch.cuda.device_count), ('eval', eval),
                 ('div_up', lambda x, y: (x + y - 1) // y)]:
    if not omegaconf.OmegaConf.has_resolver(name):
        omegaconf.OmegaConf.register_new_resolver(name, fn)
import algo, dataloader

DEVICE = 'cuda'
BATCH = 50
NFES = [1, 2, 4, 8, 16, 32, 64, 128]

/home/ivanov/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Models

In [2]:
MODELS = { 
    'MDLM': ('ArsenyIvanov/mdlm-lm1b', ['algo=mdlm']),
    'MDLM-MoE': ('ArsenyIvanov/mdlm-moe-lm1b', ['algo=emoe', 'algo.det_moe=True']),
    'E-MoE': ('ArsenyIvanov/emoe-lm1b', ['algo=emoe']),
}
COMMON = ['mode=sample_eval', 'model=small', 'model.length=128', 'data=lm1b', 'data.wrap=True',
          'trainer.devices=1', 'sampling.predictor=ancestral_cache']

with hydra.initialize_config_dir(config_dir=f'{REPO}/configs', version_base=None):
    CONFIGS = {name: hydra.compose('config', overrides=COMMON + ov) for name, (_, ov) in MODELS.items()}
tokenizer = dataloader.get_tokenizer(CONFIGS['MDLM'])


def load_model(name):
    repo, _ = MODELS[name]
    cfg = CONFIGS[name]
    model = (algo.MDLM if cfg.algo.name == 'mdlm' else algo.EMoE)(cfg, tokenizer)
    weights = load_file(hf_hub_download(repo, 'model.safetensors'))
    missing, unexpected = model.backbone.load_state_dict(weights, strict=False)
    assert not unexpected and all('rotary_emb' in k for k in missing), (missing, unexpected)
    return model.to(DEVICE).eval()


def n_params(module):
    return sum(p.numel() for p in module.parameters())


models, rows = {}, []
for name in MODELS:
    m = models[name] = load_model(name)
    bb = m.backbone
    total = n_params(bb)
    K = len(bb.blocks[0].experts) if hasattr(bb.blocks[0], 'experts') else 1
    active = total - len(bb.blocks) * (K - 1) * n_params(bb.blocks[0].experts[0]) if K > 1 else total
    rows.append((name, K, total / 1e6, active / 1e6))
params = pd.DataFrame(rows, columns=['model', 'experts / block', 'total params (M)', 'active params (M)']).set_index('model')
params.round(1)

,experts / block,total params (M),active params (M)
model,,,
MDLM,1,139.3,139.3
MDLM-MoE,8,536.1,139.4
E-MoE,8,536.1,139.4


## FLOPs of one denoising step, by layer

For one sequence of length $L=128$ with hidden size $d=768$, timestep (noise-level) conditioning size $c=128$, vocabulary $V=30522$, $B=12$ blocks and $K$ experts:

| layer | FLOPs per sequence |
|---|---|
| timestep embedding (MLP $256 \to c \to c$) | $2(256c + c^2)$ |
| adaLN ($c \to 6d$) per block | $2 \cdot c \cdot 6d$ |
| QKV and output projections, per block | $2L \cdot d \cdot 3d + 2L \cdot d^2$ |
| attention scores $QK^\top$ and $AV$, per block | $2 \cdot 2L^2 d$ |
| MLP, or the one selected expert ($d \to 4d \to d$), per block | $2 \cdot 2L \cdot 4d^2$ |
| router (MoE models), per block | $2L \cdot dK$ |
| final adaLN ($c \to 2d$) and output head ($d \to V$) | $2 \cdot c \cdot 2d + 2L \cdot dV$ |

In [3]:
def flops_by_layer(cfg, K):
    """FLOPs of one denoiser forward for one sequence."""
    L, d, c = cfg.model.length, cfg.model.hidden_size, cfg.model.cond_dim
    V, B, r = tokenizer.vocab_size, cfg.model.n_blocks, cfg.model.get('mlp_ratio', 4)
    return {
        'timestep embedding': 2 * (256 * c + c * c),
        'adaLN modulation': B * 2 * c * 6 * d,
        'QKV and output projections': B * (2 * L * d * 3 * d + 2 * L * d * d),
        'attention (QK, AV)': B * 2 * 2 * L * L * d,
        'MLP / selected expert': B * 2 * 2 * L * d * r * d,
        'router': B * 2 * L * d * K if K > 1 else 0,
        'output head': 2 * c * 2 * d + 2 * L * d * V,
    }


def measured_flops(model):
    xt = torch.full((BATCH, model.num_tokens), model.mask_index, device=DEVICE)
    sigma = torch.ones(BATCH, 1, device=DEVICE)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model.forward(xt, sigma)
    return fc.get_total_flops() / BATCH


layers = pd.DataFrame({name: flops_by_layer(CONFIGS[name], params.loc[name, 'experts / block']) for name in MODELS})
layers.loc['total'] = layers.sum()
layers.loc['total, measured'] = [measured_flops(models[name]) for name in MODELS]
print('GFLOPs per sequence, one denoising step')
(layers / 1e9).round(4)

GFLOPs per sequence, one denoising step


,MDLM,MDLM-MoE,E-MoE
timestep embedding,0.0001,0.0001,0.0001
adaLN modulation,0.0142,0.0142,0.0142
QKV and output projections,7.2478,7.2478,7.2478
"attention (QK, AV)",0.6040,0.6040,0.6040
MLP / selected expert,14.4955,14.4955,14.4955
router,0.0000,0.0189,0.0189
output head,6.0013,6.0013,6.0013
total,28.3628,28.3816,28.3816
"total, measured",28.3628,28.3816,28.3816


## FLOPs of full generation

Total FLOPs to generate one sequence of 128 token on LM1B dataset with the default sampler (`ancestral_cache`), measured with `FlopCounterMode` from Torch over a batch of 50 sequences. The sampler skips the denoiser call when no token of the batch was unmasked at a step, so for large NFE the total is slightly below (NFE + 1) denoiser calls. The first table reports GFLOPs per generated sequence, the second divides them by the sequence length (128 tokens) to give GFLOPs per generated token.

In [4]:
gen = {}
for name, m in models.items():
    for nfe in NFES:
        torch.manual_seed(0)
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            m.generate_samples(num_samples=BATCH, num_steps=nfe, eps=1e-5)
        gen.setdefault(name, {})[f'NFE={nfe}'] = fc.get_total_flops() / BATCH / 1e9
gen = pd.DataFrame(gen).T
print('GFLOPs per generated sequence')
gen.round(1)

GFLOPs per generated sequence


,NFE=1,NFE=2,NFE=4,NFE=8,NFE=16,NFE=32,NFE=64,NFE=128
MDLM,56.7,85.1,141.8,255.3,482.2,936.0,1843.6,3658.8
MDLM-MoE,56.8,85.2,141.9,255.4,482.5,936.6,1844.9,3661.4
E-MoE,56.8,85.1,141.9,255.4,482.5,936.6,1844.8,3661.2


In [5]:
L = models["MDLM"].num_tokens
gen_per_token = gen / L
print("GFLOPs per generated token")
gen_per_token.round(2)

GFLOPs per generated token


,NFE=1,NFE=2,NFE=4,NFE=8,NFE=16,NFE=32,NFE=64,NFE=128
MDLM,0.44,0.66,1.11,1.99,3.77,7.31,14.40,28.58
MDLM-MoE,0.44,0.67,1.11,2.00,3.77,7.32,14.41,28.60
E-MoE,0.44,0.67,1.11,2.00,3.77,7.32,14.41,28.60
